# Inception Network

> Based on Stanford CS230 — C4M2, Andrew Ng / DeepLearning.AI

GoogLeNet (Szegedy et al., 2014) introduced the **Inception module** — instead of choosing which filter size to use, apply them all in parallel and let the network learn which matters. The 1×1 convolution ("network in a network") keeps the computation tractable.

## Learning Objectives

1. Explain how an Inception module applies multiple filter sizes simultaneously
2. Compute how 1×1 convolutions reduce computational cost
3. Trace shapes through an Inception module
4. Understand the role of auxiliary classifiers in GoogLeNet

## The Inception Module

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 700 240" width="700" height="240" style="font-family:DejaVu Sans,Arial,sans-serif;background:#f5f5f7;border-radius:8px">
  <defs>
    <marker id="ia" markerWidth="7" markerHeight="7" refX="5" refY="3" orient="auto">
      <path d="M0,0 L0,6 L7,3 z" fill="#888"/>
    </marker>
  </defs>
  <!-- Input box -->
  <rect x="290" y="10" width="120" height="36" rx="5" fill="#5b9bd5" fill-opacity="0.15" stroke="#5b9bd5" stroke-width="1.6"/>
  <text x="350" y="32" text-anchor="middle" fill="#3a7bbf" font-size="11" font-weight="bold">Previous layer (28×28×192)</text>
  <!-- Branch boxes -->
  <!-- 1×1 -->
  <rect x="20"  y="90" width="100" height="36" rx="4" fill="#7ecba1" fill-opacity="0.15" stroke="#7ecba1" stroke-width="1.5"/>
  <text x="70"  y="111" text-anchor="middle" fill="#3d7a5e" font-size="10" font-weight="bold">1×1 Conv (64)</text>
  <!-- 1×1 → 3×3 -->
  <rect x="180" y="80"  width="100" height="28" rx="4" fill="#f4b942" fill-opacity="0.12" stroke="#f4b942" stroke-width="1.4"/>
  <rect x="180" y="118" width="100" height="28" rx="4" fill="#7ecba1" fill-opacity="0.15" stroke="#7ecba1" stroke-width="1.5"/>
  <text x="230" y="98"  text-anchor="middle" fill="#a07010" font-size="10">1×1 Conv (96)</text>
  <text x="230" y="137" text-anchor="middle" fill="#3d7a5e" font-size="10" font-weight="bold">3×3 Conv (128)</text>
  <!-- 1×1 → 5×5 -->
  <rect x="340" y="80"  width="100" height="28" rx="4" fill="#f4b942" fill-opacity="0.12" stroke="#f4b942" stroke-width="1.4"/>
  <rect x="340" y="118" width="100" height="28" rx="4" fill="#7ecba1" fill-opacity="0.15" stroke="#7ecba1" stroke-width="1.5"/>
  <text x="390" y="98"  text-anchor="middle" fill="#a07010" font-size="10">1×1 Conv (16)</text>
  <text x="390" y="137" text-anchor="middle" fill="#3d7a5e" font-size="10" font-weight="bold">5×5 Conv (32)</text>
  <!-- 3×3 MaxPool → 1×1 -->
  <rect x="510" y="80"  width="100" height="28" rx="4" fill="#c678dd" fill-opacity="0.12" stroke="#c678dd" stroke-width="1.4"/>
  <rect x="510" y="118" width="100" height="28" rx="4" fill="#7ecba1" fill-opacity="0.15" stroke="#7ecba1" stroke-width="1.5"/>
  <text x="560" y="98"  text-anchor="middle" fill="#8a3ab8" font-size="10">MaxPool 3×3</text>
  <text x="560" y="137" text-anchor="middle" fill="#3d7a5e" font-size="10" font-weight="bold">1×1 Conv (32)</text>
  <!-- Concat box -->
  <rect x="190" y="195" width="320" height="36" rx="5" fill="#e05c5c" fill-opacity="0.12" stroke="#e05c5c" stroke-width="1.6"/>
  <text x="350" y="217" text-anchor="middle" fill="#b03a3a" font-size="11" font-weight="bold">Concatenate along channels: 28×28×256</text>
  <!-- Arrows from input to branches -->
  <line x1="350" y1="46"  x2="70"  y2="89"  stroke="#aaa" stroke-width="1.2" marker-end="url(#ia)"/>
  <line x1="350" y1="46"  x2="230" y2="79"  stroke="#aaa" stroke-width="1.2" marker-end="url(#ia)"/>
  <line x1="350" y1="46"  x2="390" y2="79"  stroke="#aaa" stroke-width="1.2" marker-end="url(#ia)"/>
  <line x1="350" y1="46"  x2="560" y2="79"  stroke="#aaa" stroke-width="1.2" marker-end="url(#ia)"/>
  <!-- Arrows from branches to concat -->
  <line x1="70"  y1="126" x2="70"  y2="175" stroke="#aaa" stroke-width="1.2"/>
  <line x1="70"  y1="175" x2="190" y2="195" stroke="#aaa" stroke-width="1.2" marker-end="url(#ia)"/>
  <line x1="230" y1="146" x2="230" y2="195" stroke="#aaa" stroke-width="1.2" marker-end="url(#ia)"/>
  <line x1="390" y1="146" x2="390" y2="195" stroke="#aaa" stroke-width="1.2" marker-end="url(#ia)"/>
  <line x1="560" y1="146" x2="560" y2="175" stroke="#aaa" stroke-width="1.2"/>
  <line x1="560" y1="175" x2="510" y2="195" stroke="#aaa" stroke-width="1.2" marker-end="url(#ia)"/>
  <!-- Channel count annotations -->
  <text x="70"  y="170" text-anchor="middle" fill="#888" font-size="9">64 ch</text>
  <text x="230" y="170" text-anchor="middle" fill="#888" font-size="9">128 ch</text>
  <text x="390" y="170" text-anchor="middle" fill="#888" font-size="9">32 ch</text>
  <text x="560" y="170" text-anchor="middle" fill="#888" font-size="9">32 ch</text>
</svg>

Total output channels: $64 + 128 + 32 + 32 = 256$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': '#f5f5f7', 'axes.facecolor': '#ffffff',
    'axes.edgecolor': '#c8ccd4', 'axes.labelcolor': '#1a1d27',
    'axes.titlecolor': '#1a1d27', 'xtick.color': '#2a2e3a',
    'ytick.color': '#2a2e3a', 'grid.color': '#e0e3ea',
    'grid.linestyle': '--', 'grid.alpha': 0.5,
    'text.color': '#1a1d27', 'font.family': 'DejaVu Sans',
    'axes.titlesize': 12, 'axes.labelsize': 11,
    'legend.facecolor': '#ffffff', 'legend.edgecolor': '#c8ccd4',
    'figure.dpi': 110,
})
P = ['#5b9bd5', '#e05c5c', '#f4b942', '#7ecba1', '#56b6c2', '#c678dd']

# ---- Computation cost: direct 5×5 vs 1×1 bottleneck ----
# Input: 28×28×192, Output: 28×28×32
def compute_ops(h, w, c_in, f, c_out):
    """Number of multiplications for one conv layer."""
    return h * w * c_in * f * f * c_out

h, w = 28, 28
c_in = 192
c_out_5x5 = 32
c_bottleneck = 16

# Direct 5×5
direct = compute_ops(h, w, c_in, 5, c_out_5x5)

# Bottleneck: 1×1 then 5×5
bottle_1x1 = compute_ops(h, w, c_in,          1, c_bottleneck)
bottle_5x5 = compute_ops(h, w, c_bottleneck,  5, c_out_5x5)
bottleneck_total = bottle_1x1 + bottle_5x5

print(f"Input volume: {h}×{w}×{c_in}  →  {h}×{w}×{c_out_5x5}")
print()
print(f"Direct 5×5 conv:            {direct:>15,} multiplications")
print(f"Bottleneck 1×1 ({c_bottleneck} ch):    {bottle_1x1:>15,}")
print(f"Bottleneck 5×5:             {bottle_5x5:>15,}")
print(f"Bottleneck total:           {bottleneck_total:>15,} multiplications")
print(f"Reduction factor:           {direct/bottleneck_total:>15.1f}×")

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('1×1 Bottleneck — Computation Savings in Inception', fontsize=13, fontweight='bold')

labels = ['Direct\n5×5 conv', '1×1 bottleneck\n(16 channels)', '5×5 conv\n(after bottleneck)']
values = [direct, bottle_1x1, bottle_5x5]
colors = [P[1], P[2], P[0]]

axes[0].bar(labels, values, color=colors, alpha=0.85, width=0.5)
axes[0].set_ylabel('Number of multiplications')
axes[0].set_title('Direct vs Bottleneck Cost')
axes[0].grid(True, axis='y')
for bar, v in zip(axes[0].patches, values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2e5,
                 f'{v/1e6:.1f}M', ha='center', fontsize=9)

# Stacked: bottleneck breakdown
axes[1].bar(['Direct 5×5'], [direct],        color=P[1], alpha=0.85, label=f'Direct: {direct/1e6:.0f}M')
axes[1].bar(['Bottleneck'], [bottle_1x1],     color=P[2], alpha=0.85, label=f'1×1: {bottle_1x1/1e6:.1f}M')
axes[1].bar(['Bottleneck'], [bottle_5x5],     color=P[0], alpha=0.85, bottom=bottle_1x1, label=f'5×5: {bottle_5x5/1e6:.1f}M')
axes[1].set_ylabel('Multiplications')
axes[1].set_title(f'Bottleneck saves {direct/bottleneck_total:.0f}× compute')
axes[1].legend(fontsize=9)
axes[1].grid(True, axis='y')

plt.tight_layout()
plt.show()
